[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/10_moe_and_routing.ipynb)

# 10. Mixture-of-Experts and routing — real dispatch path

router가 expert id만 고르는 데서 끝내지 않고 **router logits → top-k → gate normalization → token dispatch → expert FFN → weighted combine → load statistics**까지 실제 계산 사슬을 본다.

이전 버전에서 Mixtral top-2는 expert id만 뽑고 실제 두 expert 출력을 합치지 않았고, DeepSeekMoE도 shared expert라는 이름만 붙어 있었다. 그 부분을 보강했다.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)


## 1. Dense FFN baseline

dense Transformer FFN에서는 모든 token이 같은 FFN parameter를 지난다. MoE는 이 FFN 위치를 여러 expert로 분해하고 router가 token별 실행 expert를 결정한다.


In [ ]:
tokens = torch.randn(
    8, 12,
    device=device,
)

dense_ffn = nn.Sequential(
    nn.Linear(12, 32),
    nn.SiLU(),
    nn.Linear(32, 12),
).to(device)

dense_output = dense_ffn(tokens)
print("dense output:", dense_output.shape)


## 2. Switch-style top-1 routing

Switch Transformer의 핵심은 token마다 expert 하나를 선택해 dense FFN 대신 sparse expert 하나만 실행하는 것이다. 실제 시스템에서는 expert capacity와 load-balancing auxiliary loss도 중요하다.


In [ ]:
num_experts = 4

router = nn.Linear(12, num_experts, bias=False).to(device)
experts = nn.ModuleList(
    [
        nn.Sequential(
            nn.Linear(12, 24),
            nn.SiLU(),
            nn.Linear(24, 12),
        ).to(device)
        for _ in range(num_experts)
    ]
)

router_logits = router(tokens)
router_probs = router_logits.softmax(dim=-1)
top1_prob, top1_expert = router_probs.max(dim=-1)

top1_output = torch.zeros_like(tokens)

for expert_id, expert in enumerate(experts):
    token_mask = top1_expert == expert_id

    if token_mask.any():
        expert_input = tokens[token_mask]
        expert_output = expert(expert_input)

        top1_output[token_mask] = (
            top1_prob[token_mask, None]
            * expert_output
        )

expert_fraction = torch.stack(
    [
        (top1_expert == expert_id).float().mean()
        for expert_id in range(num_experts)
    ]
)
mean_router_probability = router_probs.mean(dim=0)

load_balance_loss = (
    num_experts
    * torch.sum(
        expert_fraction
        * mean_router_probability
    )
)

print("top-1 expert ids:", top1_expert)
print("expert token fraction:", expert_fraction)
print("load-balance term:", load_balance_loss.item())


## 3. Mixtral-style top-2 sparse MoE

Mixtral 계열에서는 각 token이 top-2 expert를 선택하고 **선택된 두 expert의 출력 자체를 normalized routing weight로 합친다**. 이 weighted combine이 빠지면 top-2 MoE의 실제 forward가 아니다.


In [ ]:
router_logits = router(tokens)
top2_logits, top2_ids = router_logits.topk(
    k=2,
    dim=-1,
)
top2_gates = top2_logits.softmax(dim=-1)

top2_output = torch.zeros_like(tokens)

for slot in range(2):
    selected_ids = top2_ids[:, slot]
    selected_gates = top2_gates[:, slot]

    for expert_id, expert in enumerate(experts):
        token_mask = selected_ids == expert_id

        if token_mask.any():
            expert_output = expert(tokens[token_mask])
            gate = selected_gates[token_mask, None]

            top2_output[token_mask] += gate * expert_output

print("top-2 expert ids:\n", top2_ids)
print("top-2 normalized gates:\n", top2_gates)
print("combined output shape:", top2_output.shape)


## 4. Capacity and dropped/overflow tokens

분산 MoE에서는 expert마다 무한히 많은 token을 받을 수 없다. 아래는 top-1 선택 결과를 expert capacity에 맞춰 잘라내며, 왜 load balancing이 단순 통계가 아니라 실제 처리량 문제인지 보여준다.


In [ ]:
capacity = 2
accepted = torch.zeros(
    tokens.size(0),
    dtype=torch.bool,
    device=device,
)

for expert_id in range(num_experts):
    token_ids = torch.nonzero(
        top1_expert == expert_id,
        as_tuple=False,
    ).flatten()

    accepted[token_ids[:capacity]] = True

print("accepted tokens:", accepted)
print("overflow count:", int((~accepted).sum()))


## 5. DeepSeekMoE-style shared + routed experts

DeepSeekMoE의 중요한 설계 중 하나는 항상 실행되는 shared expert와 token별 routed experts를 함께 두는 것이다. 아래에서는 routed top-2 결과와 shared FFN 출력을 실제로 합친다.


In [ ]:
shared_expert = nn.Sequential(
    nn.Linear(12, 24),
    nn.SiLU(),
    nn.Linear(24, 12),
).to(device)

shared_output = shared_expert(tokens)
deepseek_style_output = shared_output + top2_output

print("shared output:", shared_output.shape)
print("shared + routed output:", deepseek_style_output.shape)


## References and provenance

**Switch Transformer** — Fedus et al. top-1 routing, sparse expert execution, capacity/load balancing의 핵심을 반영했다.

**Mixtral 8x7B** — Mistral AI report. token별 top-2 routing과 선택 expert 출력의 normalized weighted sum을 반영했다.

**DeepSeekMoE / DeepSeek-V2/V3** — fine-grained routed experts와 shared expert 계열의 아이디어를 참조했다. 이 작은 예제는 distributed expert parallelism은 생략하지만 shared path와 routed path가 실제 forward에서 합쳐지는 구조는 유지한다.
